# NB69 — Lemma-count-matched custom concepts vs WordNet, FRH vs subspace

NB68 compared WordNet and custom concept sources, but the sources differ in size: the custom
set has ~100 consensus-ranked words per concept, while the eight-language WordNet synsets
have 2–28 member lemmas. NB69 adds a third source, **`custom_top_c`**: for each concept, the
first *c* ranked custom words, where *c* is the number of WordNet member lemmas behind that
concept's WordNet frame. It separates **how many** words a concept has from **which** words.

| source | words per concept |
|---|---|
| `wordnet` | *c* WordNet member lemmas, eight languages (NB68) |
| `custom` | consensus top 100 (NB68) |
| `custom_top_c` | first *c* of the custom ranking (new) |

Everything else is NB68's frozen battery: 16 concepts, 74 directed configurations,
FRH frame blending vs pooled rank-8 subspace steering, λ ∈ {0, 0.4}, 24 steps, k=4,
bf16 Llama-3.1-8B-Instruct. Prompt p07 is excluded from analysis (its unsteered baseline
is empty); it is still generated so every batch matches NB68's eight-prompt batches, since
bf16 greedy decoding depends on batch composition.

Only `custom_top_c` is generated (2,368 continuations, 2,072 analysed). NB68's WordNet and full-custom
runs are reused after a **reproduction check** confirms the current code regenerates them
exactly, and they are rescored on the two new reference views. All runs share NB68's
baselines and are measured on the same **six views** (three sources × frame/subspace).
Cosine is `signed_mean_delta` (steered − baseline; repulsion sign-flipped); fluency is
continuation-only ΔPPL.

Default execution is the smoke battery; set `SMOKE=False` for the full run.

**`NORMALIZE_SCORE=True`** reruns the experiment with norm-normalized subspace scoring
(`‖Bᵀh‖ / ‖h‖` instead of `‖Bᵀh‖`) under the prefix `69_topc_norm`. Normalization only
changes subspace steering, so only the subspace arm is regenerated (all three sources, both
λ); FRH and the raw-score subspace runs are reused from NB68/NB69 as reference arms. A
raw-score reproduction check gates the reuse.

**`SUBSPACE_RANK=16`** (or 32) reruns the subspace arm with that many steering directions per
concept and per blend, under the prefix `69_topc_r{rank}`. Cosine is still measured on the
fixed rank-8 reference views, so ranks share one yardstick; FRH and rank-8 subspace runs are
reused, gated by the same reproduction check. The analysis includes every rank run found
on disk.

In [ ]:
# Papermill parameters
SMOKE = True
RUN_GENERATION = True
NORMALIZE_SCORE = False
SUBSPACE_RANK = 0  # 0 = NB69's rank 8; 16/32 = rank variant
OUTPUT_PREFIX = ""

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from frames.experiments.steering_comparison import freeze_json, matrix_size, read_records
from frames.experiments.topc_comparison import (
    NB68_PREFIX,
    EXCLUDED_PROMPTS,
    NEW_SOURCE,
    TopCBackend,
    make_experiment,
    reproduction_check,
    rescore_nb68,
)

from frames.experiments import topc_normalized, topc_rank

assert not (NORMALIZE_SCORE and SUBSPACE_RANK), "choose one variant"
if SUBSPACE_RANK:
    manifest = topc_rank.make_experiment(SUBSPACE_RANK, smoke=SMOKE)
    default_prefix = f"resources/69_topc_r{SUBSPACE_RANK}"
elif NORMALIZE_SCORE:
    manifest = topc_normalized.make_experiment(smoke=SMOKE)
    default_prefix = "resources/69_topc_norm"
else:
    manifest = make_experiment(smoke=SMOKE)
    default_prefix = "resources/69_topc"
prefix = OUTPUT_PREFIX or (default_prefix + ("_smoke" if SMOKE else ""))
freeze_json(f"{prefix}_manifest.json", manifest)
print(
    f"{len(manifest['configs'])} configurations × {len(manifest['prompts'])} prompts; "
    f"{matrix_size(manifest)} new continuations "
    f"({', '.join(manifest['methods'])} × {', '.join(manifest['sources'])})"
)

## Construction

*c* counts the distinct lowercased WordNet member lemmas across the eight Llama languages
that survive the loader's token filter (≤ 4 tokens) — exactly the lemmas the NB68 WordNet
frame is built from. Custom words that exceed the token limit are skipped, so each
`custom_top_c` concept has exactly *c* words. Construction fails if NB68's WordNet or
custom token rows no longer reproduce.

In [ ]:
if RUN_GENERATION:
    backend = topc_rank.RankBackend(manifest) if SUBSPACE_RANK else TopCBackend(manifest)
    construction = pd.DataFrame(
        [r for r in backend.provenance["construction"] if r["source"] == NEW_SOURCE]
    )
    display(
        construction[["concept", "canonical_synset", "c", "frame_rank", "subspace_rank", "words", "wordnet_lemmas"]]
    )
else:
    print("Generation disabled; analysis of existing results only")

## Reproduction check

Regenerates NB68 runs (first two configurations × both sources × both methods × both λ, in
NB68's eight-prompt batch) and rescores their NB68 continuations on NB68's four views. Reuse is valid only
if every text matches exactly and every cosine value matches.

In [ ]:
if RUN_GENERATION:
    if SUBSPACE_RANK:
        display(pd.DataFrame(backend.provenance["steering"]).pivot(index="concept", columns="source", values="realized_rank"))
        check = pd.DataFrame(topc_rank.reference_reproduction_check(backend, manifest))
    elif NORMALIZE_SCORE:
        check = pd.DataFrame(topc_normalized.raw_reproduction_check(backend, manifest))
    else:
        check = pd.DataFrame(reproduction_check(backend, manifest))
    display(check.groupby(["source", "method", "lambda"]).agg(same_text=("same_text", "mean"), max_cosine_gap=("max_cosine_gap", "max")))
    display(check[~check.same_text])
    assert check.same_text.all(), "Current code no longer reproduces NB68 generations"
    assert check.max_cosine_gap.max() < 1e-6, "Current code no longer reproduces NB68 cosines"
    print(f"Reproduced {len(check)} NB68 runs exactly")

## Generation and rescoring

In [ ]:
if RUN_GENERATION:
    from frames.experiments.steering_comparison import run_battery

    records = run_battery(manifest, backend, prefix)
    print(f"{len(records)} new continuations")
    if not (NORMALIZE_SCORE or SUBSPACE_RANK):  # variants are measured on all six views
        n = rescore_nb68(backend, manifest, f"{prefix}_nb68_rescore.jsonl")
        print(f"{n} NB68 continuations rescored on the {NEW_SOURCE} views")

## Results

*Own view* scores each run on its own source's reference of the method's kind (frame for FRH,
subspace for subspace steering). The source comparison also uses fixed common views, since
the own view changes with the source.

In [ ]:
METHOD_COLOR = {"frh_blend": "#2a78d6", "subspace": "#eb6834"}
if NORMALIZE_SCORE:
    METHOD_COLOR["subspace_norm"] = "#1baf7a"
suffix = "_smoke" if SMOKE else ""
rank_prefixes = [
    f"resources/69_topc_r{r}{suffix}" for r in (16, 32)
    if SUBSPACE_RANK and Path(f"resources/69_topc_r{r}{suffix}.jsonl").exists()
]
for r, color in zip((16, 32), ("#1baf7a", "#e87ba4")):
    if f"resources/69_topc_r{r}{suffix}" in rank_prefixes:
        METHOD_COLOR[f"subspace_r{r}"] = color
SOURCES = ["wordnet", "custom_top_c", "custom"]
SOURCE_STYLE = {"wordnet": ":", "custom_top_c": "-", "custom": "--"}
SOURCE_MARKER = {"wordnet": "o", "custom_top_c": "D", "custom": "s"}
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.3)

configs = {c["id"] for c in manifest["configs"]}
prompts = {p["id"] for p in manifest["prompts"]} - set(EXCLUDED_PROMPTS)
nb69_prefix = "resources/69_topc" + suffix
if SUBSPACE_RANK:
    all_runs = topc_rank.load_runs(rank_prefixes, nb69_prefix)
elif NORMALIZE_SCORE:
    all_runs = topc_normalized.load_runs(prefix, nb69_prefix)
else:
    rescore = {r["run_id"]: r["cosine"] for r in read_records(f"{prefix}_nb68_rescore.jsonl")}
    nb68 = [
        r for r in read_records(f"{NB68_PREFIX}.jsonl")
        if r["config"]["id"] in configs and r["prompt"]["id"] in prompts
    ]
    assert len(rescore) == len(nb68), "NB68 rescoring is incomplete"
    all_runs = [
        {**r, "metrics": {**r["metrics"], "cosine": {**r["metrics"]["cosine"], **rescore[r["run_id"]]}}}
        for r in nb68
    ] + [r for r in read_records(f"{prefix}.jsonl") if r["prompt"]["id"] in prompts]
all_runs = [r for r in all_runs if r["config"]["id"] in configs and r["prompt"]["id"] in prompts]
baseline_text = dict(zip(
    (p["id"] for p in json.load(open(f"{prefix}_baselines.json"))["prompts"]),
    json.load(open(f"{prefix}_baselines.json"))["continuations"],
))

rows = []
for r in all_runs:
    cosine = r["metrics"]["cosine"]
    f = r["metrics"]["fluency"]
    own = f'{r["source"]}_{"frame" if r["method"] == "frh_blend" else "subspace"}'
    for view, m in cosine.items():
        s = m["summary"] if m["valid"] else {}
        rows.append(dict(
            run_id=r["run_id"], source=r["source"], method=r["method"], lam=r["lambda"],
            config=r["config"]["id"], kind=r["config"]["kind"], units=r["config"]["unit_count"],
            prompt=r["prompt"]["id"], view=view, own=view == own,
            cos_valid=m["valid"], signed=s.get("signed_mean_delta"),
            all_improved=s.get("all_requested_improved"),
            ppl_valid=f["valid"], delta_ppl=f["delta_ppl"],
            empty=r["metrics"]["continuation_tokens"] == 0,
            short=r["metrics"]["continuation_tokens"] < 12,
            repetitive=bool(re.search(r"(.{1,4})\1{5,}", r["continuation"])),
            same_as_baseline=r["continuation"] == baseline_text[r["prompt"]["id"]],
        ))
df = pd.DataFrame(rows)
print(df.drop_duplicates("run_id").groupby(["source", "method", "lam"]).size().unstack())
print("invalid cosine rows:", (~df.cos_valid).sum(), "| invalid PPL runs:", (~df.drop_duplicates("run_id").ppl_valid).sum())

### Headline: own-view cosine gain vs fluency cost

In [ ]:
v = df[df.own & df.cos_valid & df.ppl_valid]
agg = v.groupby(["method", "source", "lam"]).agg(
    cos=("signed", "mean"), cos_se=("signed", "sem"),
    ppl_med=("delta_ppl", "median"),
    ppl_q1=("delta_ppl", lambda x: x.quantile(0.25)),
    ppl_q3=("delta_ppl", lambda x: x.quantile(0.75)),
    all_improved=("all_improved", "mean"), n=("signed", "size"),
).reset_index()
agg.to_csv(f"{prefix}_summary.csv", index=False)
display(agg.round(4))

fig, ax = plt.subplots(figsize=(8, 5.5))
for (m, s), g in agg.groupby(["method", "source"]):
    g = g.sort_values("lam")
    ax.plot(g.ppl_med, g.cos, SOURCE_STYLE[s], color=METHOD_COLOR[m], lw=2, alpha=0.6)
    ax.errorbar(
        g.ppl_med, g.cos, yerr=1.96 * g.cos_se,
        xerr=[g.ppl_med - g.ppl_q1, g.ppl_q3 - g.ppl_med],
        fmt=SOURCE_MARKER[s], color=METHOD_COLOR[m], ms=8, capsize=3, lw=1, label=f"{m} · {s}",
    )
    for _, r in g.iterrows():
        ax.annotate(f"λ={r.lam:g}", (r.ppl_med, r.cos), textcoords="offset points", xytext=(6, 4), fontsize=9, color="0.3")
ax.axhline(0, color="0.5", lw=0.8)
ax.set_xlabel("median ΔPPL (IQR bars)  ← more fluent")
ax.set_ylabel("mean own-view signed Δcos (95% CI)")
ax.set_title("NB69: gain vs fluency cost by concept source")
ax.legend(frameon=False, fontsize=9)
fig.savefig(f"{prefix}_gain_vs_ppl.png", dpi=300, bbox_inches="tight")
plt.show()

### Source effects on common views

Paired on method, λ, configuration and prompt. **Size effect**: `custom − custom_top_c`
(same ranking, more words). **Word-choice effect**: `custom_top_c − wordnet` (same number
of words, different words). Each is shown on all six views.

In [ ]:
VIEWS = [f"{s}_{k}" for s in SOURCES for k in ("frame", "subspace")]
p = df[df.cos_valid].pivot_table(index=["method", "lam", "config", "prompt", "view"], columns="source", values="signed")
contrasts = {"size: custom − custom_top_c": ("custom", "custom_top_c"), "words: custom_top_c − wordnet": ("custom_top_c", "wordnet")}
rng = np.random.default_rng(0)
out = []
for name, (a, b) in contrasts.items():
    d = (p[a] - p[b]).dropna().rename("diff").reset_index()
    for (m, lam, view), g in d.groupby(["method", "lam", "view"]):
        boot = rng.choice(g["diff"].to_numpy(), (2000, len(g))).mean(1)
        out.append(dict(contrast=name, method=m, lam=lam, view=view, mean=g["diff"].mean(),
                        lo=np.percentile(boot, 2.5), hi=np.percentile(boot, 97.5),
                        frac_positive=(g["diff"] > 0).mean(), n=len(g)))
contrast = pd.DataFrame(out)
contrast.to_csv(f"{prefix}_source_contrasts.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
lams = sorted(df.lam.unique())
for ax, name in zip(axes, contrasts):
    for i, (m, lam) in enumerate([(m, lam) for m in METHOD_COLOR for lam in lams]):
        g = contrast[(contrast.contrast == name) & (contrast.method == m) & (contrast.lam == lam)].set_index("view").reindex(VIEWS)
        y = np.arange(len(VIEWS)) + (i - 1.5) * 0.18
        ax.errorbar(g["mean"], y, xerr=[g["mean"] - g.lo, g.hi - g["mean"]], fmt="o" if lam == 0 else "D",
                    color=METHOD_COLOR[m], alpha=1 if lam else 0.55, capsize=2, ms=5, label=f"{m} · λ={lam:g}")
    ax.axvline(0, color="0.5", lw=0.8)
    ax.set_yticks(range(len(VIEWS)))
    ax.set_yticklabels(VIEWS)
    ax.set_title(name)
    ax.set_xlabel("paired signed Δcos difference (bootstrap 95% CI)")
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(f"{prefix}_source_contrasts.png", dpi=300, bbox_inches="tight")
plt.show()

pp = df.drop_duplicates("run_id").query("ppl_valid").pivot_table(index=["method", "lam", "config", "prompt"], columns="source", values="delta_ppl")
display(pd.DataFrame({name: (pp[a] - pp[b]).groupby(level=["method", "lam"]).median() for name, (a, b) in contrasts.items()}).round(2).rename_axis(columns="median paired ΔPPL difference"))

### Per concept: does the WordNet gap depend on c? (single-concept runs, own view)

In [ ]:
single = df[df.own & df.cos_valid & (df.kind == "single")].copy()
single["concept"] = single.config.str.removeprefix("n1-")
cs = pd.read_json(f"{prefix}_construction.json", typ="series")["construction"]
c_of = {r["concept"]: r["c"] for r in cs if r["source"] == NEW_SOURCE}
per = single.groupby(["method", "lam", "concept", "source"]).signed.mean().unstack("source").reset_index()
per["c"] = per.concept.map(c_of)
per.to_csv(f"{prefix}_per_concept.csv", index=False)
fig, axes = plt.subplots(1, len(METHOD_COLOR), figsize=(6.5 * len(METHOD_COLOR), 4.8), sharey=True)
for ax, m in zip(axes, METHOD_COLOR):
    g = per[(per.method == m) & (per.lam == 0.4)].sort_values("c")
    for s in SOURCES:
        ax.plot(g.c, g[s], SOURCE_MARKER[s], color=METHOD_COLOR[m], alpha={"wordnet": 0.35, "custom_top_c": 1, "custom": 0.65}[s], ms=7, label=s)
    for _, r in g.iterrows():
        ax.annotate(r.concept, (r.c, r.custom_top_c), textcoords="offset points", xytext=(4, 3), fontsize=7, color="0.35")
    ax.axhline(0, color="0.5", lw=0.8)
    ax.set_xlabel("c (WordNet member lemmas)")
    ax.set_title(f"{m} · λ=0.4")
axes[0].set_ylabel("own-view Δcos on target")
axes[0].legend(frameon=False)
fig.tight_layout()
fig.savefig(f"{prefix}_per_concept_c.png", dpi=300, bbox_inches="tight")
plt.show()

### Scaling with steering units (own view)

In [ ]:
g = df[df.own & df.cos_valid].groupby(["lam", "method", "source", "units"]).signed.agg(["mean", "sem"]).reset_index()
fig, axes = plt.subplots(1, len(lams), figsize=(13, 4.5), sharey=True)
for ax, lam in zip(axes, lams):
    for (m, s), d in g[g.lam == lam].groupby(["method", "source"]):
        ax.errorbar(d.units, d["mean"], yerr=1.96 * d["sem"], ls=SOURCE_STYLE[s], marker=SOURCE_MARKER[s],
                    color=METHOD_COLOR[m], lw=2, ms=6, capsize=3, label=f"{m} · {s}")
    ax.axhline(0, color="0.5", lw=0.8)
    ax.set_xticks([1, 2, 3, 4])
    ax.set_xlabel("steering units")
    ax.set_title(f"λ={lam:g}")
axes[0].set_ylabel("own-view signed Δcos")
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

### Qualitative gallery

Side-by-side continuations for all three sources and both methods (median and strongest
prompts), written to `{prefix}_gallery.md`.

In [ ]:
if not (NORMALIZE_SCORE or SUBSPACE_RANK):
    from frames.experiments.topc_gallery import write_gallery

    print(write_gallery(prefix))

### Output health

Share of runs that stop early (< 12 of 25 tokens), repeat a short unit six or more times,
or are byte-identical to the unsteered baseline (steering had no effect), with own-view
Δcos over the remaining clean runs.

In [ ]:
runs = df[df.own].copy()
runs["degenerate"] = runs.short | runs.repetitive
health = runs.groupby(["method", "source", "lam"]).agg(
    short=("short", "mean"), repetitive=("repetitive", "mean"),
    same_as_baseline=("same_as_baseline", "mean"),
    cos_all=("signed", "mean"),
    cos_clean=("signed", lambda x: x[~runs.loc[x.index, "degenerate"]].mean()),
).round(4)
health.to_csv(f"{prefix}_health.csv")
display(health)